In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.models as models

from PIL import Image
from torchvision import transforms

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [ ]:
model = models.resnet50(
    weights=None
)

model.fc = nn.Linear(
    model.fc.in_features,
    3
)

In [ ]:
checkpoint_path = "../models/best_resnet50.pth"
model.load_state_dict(

    torch.load(
        checkpoint_path,
        map_location=device
    )
)

model = model.to(device)

model.eval()

print("ResNet50 model loaded successfully.")

In [ ]:
GRADCAM_MASK_DIR = \
    "../outputs/gradcam/masks"

VIT_MASK_DIR = \
    "../outputs/vit/attention_rollout/masks"

TEST_DIR = \
    "../data/COVID_19_dataset/test"

In [ ]:
class_to_idx = {

    "COVID": 0,

    "Normal": 1,

    "Viral Pneumonia": 2
}

In [ ]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[0.5159]*3,
        std=[0.2280]*3
    )
])

In [ ]:
def load_image(image_path):

    image = Image.open(
        image_path
    ).convert("RGB")

    tensor = transform(
        image
    )

    return tensor

In [ ]:
def compute_entropy(mask):

    mask = mask.flatten()

    mask = mask / (
        mask.sum() + 1e-8
    )

    entropy = -np.sum(

        mask * np.log(
            mask + 1e-8
        )
    )

    return entropy

In [ ]:
def evaluate_entropy(mask_dir):

    correct_entropy = []
    wrong_entropy = []

    for file in os.listdir(mask_dir):

        if not file.endswith(".npy"):
            continue

        if (
            "true_" not in file or
            "_pred_" not in file
        ):
            continue

        mask = np.load(
            os.path.join(
                mask_dir,
                file
            )
        )

        entropy = compute_entropy(mask)

        parts = file.replace(
            ".npy",
            ""
        ).split("_")

        try:

            true_class = parts[2]

            pred_class = parts[4]

        except IndexError:

            continue

        if true_class == pred_class:

            correct_entropy.append(
                entropy
            )

        else:

            wrong_entropy.append(
                entropy
            )

    return (
        correct_entropy,
        wrong_entropy
    )

In [ ]:
grad_correct, grad_wrong = \
    evaluate_entropy(
        GRADCAM_MASK_DIR
)

vit_correct, vit_wrong = \
    evaluate_entropy(
        VIT_MASK_DIR
)

In [ ]:
print(
    "GradCAM Correct Mean:",
    np.mean(grad_correct)
)

print(
    "GradCAM Wrong Mean:",
    np.mean(grad_wrong)
)

print(
    "ViT Correct Mean:",
    np.mean(vit_correct)
)

print(
    "ViT Wrong Mean:",
    np.mean(vit_wrong)
)

In [ ]:
plt.figure(figsize=(10,6))

plt.boxplot([

    grad_correct,
    grad_wrong,

    vit_correct,
    vit_wrong

])

plt.xticks(

    [1,2,3,4],

    [
        "GradCAM Correct",
        "GradCAM Wrong",

        "ViT Correct",
        "ViT Wrong"
    ]
)

plt.ylabel("Entropy")

plt.title(
    "Entropy Comparison"
)

plt.show()

In [ ]:
def perturb_image(

    img,
    mask,

    mode='deletion',

    step=0.1
):

    img = img.clone()

    c, h, w = img.shape

    flat_mask = mask.flatten()

    indices = np.argsort(
        flat_mask
    )[::-1].copy()

    num_pixels = int(
        step * len(indices)
    )

    selected = indices[:num_pixels]

    perturbed = img.view(c, -1)

    if mode == 'deletion':

        perturbed[:, selected] = 0

    elif mode == 'insertion':

        original = img.view(c,-1).clone()

        perturbed[:] = 0

        perturbed[:, selected] = \
            original[:, selected]

    return perturbed.view(c,h,w)

In [ ]:
def insertion_deletion_curve(

    model,
    image,
    mask,

    target_class
):

    deletion_scores = []
    insertion_scores = []

    steps = np.linspace(
        0.1,
        1.0,
        10
    )

    for step in steps:

        deleted = perturb_image(
            image,
            mask,
            mode='deletion',
            step=step
        )

        inserted = perturb_image(
            image,
            mask,
            mode='insertion',
            step=step
        )

        with torch.no_grad():

            del_out = model(
                deleted.unsqueeze(0).to(device)
            )

            ins_out = model(
                inserted.unsqueeze(0).to(device)
            )

            del_prob = torch.softmax(
                del_out,
                dim=1
            )[0, target_class].item()

            ins_prob = torch.softmax(
                ins_out,
                dim=1
            )[0, target_class].item()

        deletion_scores.append(
            del_prob
        )

        insertion_scores.append(
            ins_prob
        )

    return (
        deletion_scores,
        insertion_scores
    )

In [ ]:
sample_mask_file = os.listdir(
    GRADCAM_MASK_DIR
)[0]

print(sample_mask_file)

In [ ]:
sample_mask = np.load(

    os.path.join(
        GRADCAM_MASK_DIR,
        sample_mask_file
    )
)

In [ ]:
parts = sample_mask_file.replace(
    ".npy",
    ""
).split("_")

true_class = parts[2]

target_class = class_to_idx[
    true_class
]

In [ ]:
image_candidates = os.listdir(

    os.path.join(
        TEST_DIR,
        true_class
    )
)

image_path = os.path.join(

    TEST_DIR,

    true_class,

    image_candidates[0]
)

print(image_path)

In [ ]:
sample_image = load_image(
    image_path
)

In [ ]:
del_scores, ins_scores = \
    insertion_deletion_curve(

        model,

        sample_image,
        sample_mask,

        target_class
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    del_scores,
    label='Deletion'
)

plt.plot(
    ins_scores,
    label='Insertion'
)

plt.xlabel(
    "Perturbation Step"
)

plt.ylabel(
    "Confidence"
)

plt.title(
    "Insertion / Deletion Curve"
)

plt.legend()

plt.show()

In [ ]:
def compute_aopc(

    model,
    image,
    mask,

    target_class
):

    original_output = model(

        image.unsqueeze(0).to(device)
    )

    original_prob = torch.softmax(

        original_output,
        dim=1

    )[0, target_class].item()

    drops = []

    steps = np.linspace(
        0.1,
        1.0,
        10
    )

    for step in steps:

        deleted = perturb_image(
            image,
            mask,
            mode='deletion',
            step=step
        )

        output = model(
            deleted.unsqueeze(0).to(device)
        )

        prob = torch.softmax(
            output,
            dim=1
        )[0, target_class].item()

        drops.append(
            original_prob - prob
        )

    return np.mean(drops)

In [ ]:
gradcam_aopc = compute_aopc(

    model,

    sample_image,
    sample_mask,

    target_class
)

print(
    "GradCAM AOPC:",
    gradcam_aopc
)

In [ ]:
vit_mask_file = os.listdir(
    VIT_MASK_DIR
)[0]

vit_mask = np.load(

    os.path.join(
        VIT_MASK_DIR,
        vit_mask_file
    )
)

In [ ]:
vit_aopc = compute_aopc(

    model,

    sample_image,
    vit_mask,

    target_class
)

print(
    "ViT Rollout AOPC:",
    vit_aopc
)

In [ ]:
results = {

    "Method":[
        "GradCAM",
        "ViT Rollout"
    ],

    "Entropy":[
        np.mean(grad_correct),
        np.mean(vit_correct)
    ],

    "AOPC":[
        gradcam_aopc,
        vit_aopc
    ]
}

df = pd.DataFrame(
    results
)

df

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

SAVE_DIR = "../outputs/comparisons"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

# -----------------------------------
# SAVE CSV
# -----------------------------------

curve_df = pd.DataFrame({

    "step": list(
        range(len(del_scores))
    ),

    "deletion": del_scores,

    "insertion": ins_scores
})

csv_path = os.path.join(

    SAVE_DIR,

    "insertion_deletion_scores.csv"
)

curve_df.to_csv(
    csv_path,
    index=False
)

print(f"Saved CSV at: {csv_path}")

# -----------------------------------
# SAVE PLOT
# -----------------------------------

plt.figure(figsize=(8,6))

plt.plot(

    del_scores,

    label="Deletion"
)

plt.plot(

    ins_scores,

    label="Insertion"
)

plt.xlabel("Perturbation Step")

plt.ylabel("Confidence")

plt.title(
    "Insertion / Deletion Curve"
)

plt.legend()

plot_path = os.path.join(

    SAVE_DIR,

    "insertion_deletion_curve.png"
)

plt.savefig(

    plot_path,

    dpi=300,

    bbox_inches="tight"
)

plt.show()

print(f"Saved plot at: {plot_path}")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

SAVE_DIR = "../outputs/comparisons"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

# -----------------------------------
# SAVE CSV
# -----------------------------------

insertion_df = pd.DataFrame({

    "step": list(
        range(len(ins_scores))
    ),

    "confidence": ins_scores
})

csv_path = os.path.join(

    SAVE_DIR,

    "insertion_scores.csv"
)

insertion_df.to_csv(
    csv_path,
    index=False
)

print(f"Saved CSV at: {csv_path}")

# -----------------------------------
# PLOT
# -----------------------------------

plt.figure(figsize=(8,6))

plt.plot(
    ins_scores,
    linewidth=2
)

plt.xlabel("Perturbation Step")

plt.ylabel("Confidence")

plt.title(
    "Insertion Curve"
)

plot_path = os.path.join(

    SAVE_DIR,

    "insertion_curve.png"
)

plt.savefig(

    plot_path,

    dpi=300,

    bbox_inches="tight"
)

plt.show()

print(f"Saved plot at: {plot_path}")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

SAVE_DIR = "../outputs/comparisons"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

# -----------------------------------
# SAVE CSV
# -----------------------------------

deletion_df = pd.DataFrame({

    "step": list(
        range(len(del_scores))
    ),

    "confidence": del_scores
})

csv_path = os.path.join(

    SAVE_DIR,

    "deletion_scores.csv"
)

deletion_df.to_csv(
    csv_path,
    index=False
)

print(f"Saved CSV at: {csv_path}")

# -----------------------------------
# PLOT
# -----------------------------------

plt.figure(figsize=(8,6))

plt.plot(
    del_scores,
    linewidth=2
)

plt.xlabel("Perturbation Step")

plt.ylabel("Confidence")

plt.title(
    "Deletion Curve"
)

plot_path = os.path.join(

    SAVE_DIR,

    "deletion_curve.png"
)

plt.savefig(

    plot_path,

    dpi=300,

    bbox_inches="tight"
)

plt.show()

print(f"Saved plot at: {plot_path}")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

SAVE_DIR = "../outputs/comparisons"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

# -----------------------------------
# SAVE CSV
# -----------------------------------

aopc_df = pd.DataFrame({

    "method": [
        "GradCAM",
        "ViT Rollout"
    ],

    "aopc": [
        gradcam_aopc,
        vit_aopc
    ]
})

csv_path = os.path.join(

    SAVE_DIR,

    "aopc_scores.csv"
)

aopc_df.to_csv(
    csv_path,
    index=False
)

print(f"Saved CSV at: {csv_path}")

# -----------------------------------
# SAVE BAR PLOT
# -----------------------------------

plt.figure(figsize=(6,6))

plt.bar(

    ["GradCAM", "ViT Rollout"],

    [gradcam_aopc, vit_aopc]
)

plt.ylabel("AOPC")

plt.title(
    "AOPC Comparison"
)

plot_path = os.path.join(

    SAVE_DIR,

    "aopc_comparison.png"
)

plt.savefig(

    plot_path,

    dpi=300,

    bbox_inches="tight"
)

plt.show()

print(f"Saved plot at: {plot_path}")